# 03 · Policy Evaluation and Generalization
**Time: 40–50 minutes.**

## Learning Objectives
Explain closed-loop inference, compare matched initial states, and interpret a position-shift test without overstating generalization.

## Background
A closed-loop policy receives a new observation after every action. Small prediction errors can change later observations. Notebook 02 already executed both policies; this notebook reads those exact results. It does not rerun the simulator or choose a favorable test set.

## Step-by-step Instructions
### Step 1 — Load the current controlled experiment
Run Notebook 02 first. Restarting this notebook loads the group's existing experiment and keeps all previous rollouts.


In [ ]:
from pathlib import Path
import os, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
from manipulation_lab.teaching import read_json, markdown_table
GROUP = os.environ.get("LAB_GROUP", "group_01")  # Use the group assigned by your instructor.
RUN = workspace(GROUP)
EXPERIMENT = stage_path(RUN, "experiment")
experiment = read_json(EXPERIMENT / "experiment.json")
assert experiment["complete"], "Complete both learning-rate runs in Notebook 02 first."
primary = experiment["variants"][experiment["primary_variant"]]
TRAINING = EXPERIMENT / primary["training"]
EVALUATION = EXPERIMENT / primary["evaluation"]


### Step 2 — Inspect the initial-position distribution
Training points are the first can position of each of the **160 training demonstrations**. Evaluation points are the actual logged simulator initial states. Lines connect matched standard and shifted cases. Coordinates are world XY in meters, Z upward.

**Position Shift is not a certified out-of-distribution (OOD) test.** The shift adds 0.04 m along world X and may still overlap training support. A two-dimensional scatter plot does not establish support in the full observation space.


In [ ]:
display(Image(filename=str(EVALUATION / "initial_positions.png")))
print("Dataset SHA256:", read_json(TRAINING / "dataset_inspection.json")["dataset_sha256"])


### Step 3 — Compare each matched case
Both conditions use the same held-out demonstration state and seed, except for the can translation. Eligible starts are selected before policy execution so shifted starts remain in the source bin. Compare both environment success and **stable released placement**.

Stable placement requires the can in the target region, an open gripper and no contact, followed by at least 1 s of continued policy execution with displacement ≤0.01 m and speed ≤0.05 m/s. Merely lifting the can is not success. Full thresholds are saved in each evaluation configuration. Five cases produce a coarse estimate, not a reliable ranking of small differences.


In [ ]:
from manipulation_lab.teaching import paired_cases
for variant, paths in experiment["variants"].items():
    evaluation = EXPERIMENT / paths["evaluation"]
    results = read_json(evaluation / "summary.json")
    display(Markdown("### " + variant + " · learning rate " + str(paths["learning_rate"])))
    print("Checkpoint SHA256:", results["checkpoint_sha256"])
    display(Image(filename=str(evaluation / "success_rate_plot.png")))
    display(Markdown(markdown_table(paired_cases(results), [
        "source_episode", "seed", "standard_environment_success", "shifted_environment_success",
        "standard_stable_success", "shifted_stable_success", "standard_steps", "shifted_steps",
        "standard_error", "shifted_error"])))
    print("Saved paired cases:", evaluation / "paired_results.csv")


### Step 4 — Watch actual closed-loop rollouts
These videos use the preselected primary model, `lr_a`. If no success or no failure exists, the corresponding video is absent. Do not substitute a demonstration and label it as a policy rollout.


In [ ]:
for filename in ["successful_rollout.mp4", "failed_rollout.mp4"]:
    path = EVALUATION / filename
    if path.exists():
        display(Markdown("**Primary student model lr_a · " + filename + "**"))
        display(Video(str(path), embed=True))
    else:
        print(filename, "is absent: no completed episode of this type.")
REFERENCE_ROOT = Path(os.environ.get("LAB_REFERENCE_ROOT", ROOT / "checkpoints"))
reference = REFERENCE_ROOT / "reference_evaluation/summary.json"
if reference.is_file():
    reference_results = read_json(reference)
    assert reference_results["policy_role"] == "instructor_reference"
    print("Separate instructor model (not student scores):", reference_results["conditions"])
else:
    print("Optional reference not prepared. Instructor: see docs/server_setup.md and prepare_reference.py --train.")


## Expected Outputs
Two success-rate plots, two paired-case tables and CSVs, an initial-position scatter plot and available success/failure videos. Each result identifies its checkpoint. All evidence was saved by Notebook 02.

## Questions for Students
**Q4. How does changing the initial object position affect the learned policy?** Cite at least one paired case and the coverage plot. Distinguish observed behavior from a proposed explanation. Explain why this experiment does not prove strict OOD generalization.


In [ ]:
Q4 = ""
answer_path = RUN / "discussion_answers.json"
answers = read_json(answer_path) if answer_path.exists() else {}
answers["q4"] = Q4
write_json(answer_path, answers)


## Completion Checklist
- [ ] I inspected actual initial states and paired results for both policies.
- [ ] I distinguished environment success from stable released placement.
- [ ] I watched available policy videos and retained unsuccessful cases.
- [ ] I saved Q4 with the Position Shift limitation. Continue to Notebook 04.
